In [20]:
!nvidia-smi

Sat May 23 05:14:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
from google.colab import files

uploaded = files.upload()

Saving English-Mizo Traning Data 2026.xlsx to English-Mizo Traning Data 2026 (1).xlsx


In [22]:
import pandas as pd

file_name = "English-Mizo Traning Data 2026.xlsx"

df = pd.read_excel(file_name)

print(df.head())
print("\nColumns:\n", df.columns)
print("\nTotal Rows:", len(df))

  Now therefore ,  my son ,  obey my voice .  Arise ,  flee to Laban ,  my brother ,  in Haran .   \
0  He is the Lord our God ; his judgments are in ...                                                
1                                     Come back soon                                                
2  Then they come back by boat to Caesareʹa, and ...                                                
3  Of the tribe of Benjamin ,  Elidad the son of ...                                                
4                                  dilemma ( dilema                                                 

  Chuvângin ,    ka thu hi ngaithla teh .  Haran khuaah ka nuṭa Labana hnênah tlânchhe daih la ,      
0  Ani chu Lalpa kan Pathian chu a ni a ; A thupê...                                                  
1                               Lo kir thuai ang che                                                  
2  Tin lawngin Kaisaria-ah an kir a ,  Antiokei-a...                                

In [23]:
import pandas as pd

# Load again
df = pd.read_excel("English-Mizo Traning Data 2026.xlsx")

# CHANGE THESE if your column names are different
english_col = df.columns[0]
mizo_col = df.columns[1]

# Keep only needed columns
df = df[[english_col, mizo_col]]

# Rename columns
df.columns = ["en", "lus"]

# Remove empty rows
df = df.dropna()

# Convert to string
df["en"] = df["en"].astype(str)
df["lus"] = df["lus"].astype(str)

# Strip spaces
df["en"] = df["en"].str.strip()
df["lus"] = df["lus"].str.strip()

# Remove duplicate sentence pairs
df = df.drop_duplicates()

# Remove very short bad rows
df = df[
    (df["en"].str.len() > 1) &
    (df["lus"].str.len() > 1)
]

print(df.head())
print("\nTotal Clean Rows:", len(df))

                                                  en  \
0  He is the Lord our God ; his judgments are in ...   
1                                     Come back soon   
2  Then they come back by boat to Caesareʹa, and ...   
3  Of the tribe of Benjamin ,  Elidad the son of ...   
4                                   dilemma ( dilema   

                                                 lus  
0  Ani chu Lalpa kan Pathian chu a ni a ; A thupê...  
1                               Lo kir thuai ang che  
2  Tin lawngin Kaisaria-ah an kir a ,  Antiokei-a...  
3      Benjamina hnam aṭangin Kislona fapa Elidada ;  
4                   a ngaihna hre lova inkhapthlukna  

Total Clean Rows: 49794


In [24]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

print(dataset)
print(dataset[0])

Dataset({
    features: ['en', 'lus', '__index_level_0__'],
    num_rows: 49794
})
{'en': 'He is the Lord our God ; his judgments are in all the earth .', 'lus': 'Ani chu Lalpa kan Pathian chu a ni a ; A thupêkte chu khawvêl pum huap a ni .', '__index_level_0__': 0}


In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model and tokenizer loaded successfully.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model and tokenizer loaded successfully.


In [26]:
from transformers import NllbTokenizer

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = NllbTokenizer.from_pretrained(model_name)

print("Tokenizer loaded correctly.")

Tokenizer loaded correctly.


In [27]:
print(type(tokenizer))

<class 'transformers.models.nllb.tokenization_nllb.NllbTokenizer'>


In [28]:
print(tokenizer)

NllbTokenizer(name_or_path='facebook/nllb-200-distilled-600M', vocab_size=256204, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	256001: AddedToken("ace_Arab", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	256002: AddedToken("ace_Latn", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	256003: AddedToken("

In [29]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['en', 'lus', '__index_level_0__'],
    num_rows: 44814
})
Dataset({
    features: ['en', 'lus', '__index_level_0__'],
    num_rows: 4980
})


In [30]:
max_length = 128

source_lang = "eng_Latn"
target_lang = "lus_Latn"

tokenizer.src_lang = source_lang

def preprocess_function(examples):

    inputs = examples["en"]
    targets = examples["lus"]

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    return model_inputs

In [31]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

print(tokenized_train[0])

Map:   0%|          | 0/44814 [00:00<?, ? examples/s]

Map:   0%|          | 0/4980 [00:00<?, ? examples/s]

{'en': 'The children of Israel went into the midst of the sea on the dry ground ,  and the waters were a wall to them on their right hand ,  and on their left .', 'lus': 'Israel fate chu tuipui mawng leichârah chuan an kal ta a ,    an ding lam leh vei lamah tui chu bang angin a ding tawn a .', '__index_level_0__': 42013, 'input_ids': [256047, 1617, 34253, 452, 4992, 38406, 9174, 349, 4729, 148, 452, 349, 21175, 281, 349, 96266, 100221, 146, 540, 349, 204641, 8444, 9, 33844, 202, 7197, 281, 8334, 13119, 4636, 146, 540, 281, 8334, 37814, 81, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [32]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 1,404,497,920 || trainable%: 0.1680


In [33]:
!pip uninstall -y torchao

In [34]:
!pip install -q transformers datasets sentencepiece accelerate peft sacrebleu evaluate openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


In [35]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Loaded successfully")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded successfully


In [38]:
import pandas as pd
from datasets import Dataset
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = NllbTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# LOAD EXCEL
df = pd.read_excel("English-Mizo Traning Data 2026.xlsx")

# GET COLUMNS
english_col = df.columns[0]
mizo_col = df.columns[1]

# KEEP ONLY 2 COLUMNS
df = df[[english_col, mizo_col]]

# RENAME
df.columns = ["en", "lus"]

# REMOVE EMPTY
df = df.dropna()

# FORCE STRING CONVERSION
df["en"] = df["en"].astype(str)
df["lus"] = df["lus"].astype(str)

# CLEAN SPACES
df["en"] = df["en"].str.strip()
df["lus"] = df["lus"].str.strip()

# REMOVE DUPLICATES
df = df.drop_duplicates()

# REMOVE VERY SHORT BAD ROWS
df = df[
    (df["en"].str.len() > 1) &
    (df["lus"].str.len() > 1)
]

# CREATE DATASET
dataset = Dataset.from_pandas(df)

# TRAIN TEST SPLIT
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("Loaded successfully")
print(train_dataset)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded successfully
Dataset({
    features: ['en', 'lus', '__index_level_0__'],
    num_rows: 44814
})


In [39]:
max_length = 128

tokenizer.src_lang = "eng_Latn"

def preprocess_function(examples):

    model_inputs = tokenizer(
        examples["en"],
        text_target=examples["lus"],
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    return model_inputs

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

print("Tokenization complete")

Map:   0%|          | 0/44814 [00:00<?, ? examples/s]

Map:   0%|          | 0/4980 [00:00<?, ? examples/s]

Tokenization complete


In [40]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,359,296 || all params: 1,404,497,920 || trainable%: 0.1680


In [43]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-mizo",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_steps=100,
    report_to="none"
)

print("Training arguments ready")

Training arguments ready


In [45]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer ready")

Trainer ready


In [46]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,6.527380,6.417078
2,6.509579,6.391196
3,6.497032,6.386651


TrainOutput(global_step=33612, training_loss=6.669971419078714, metrics={'train_runtime': 6452.8432, 'train_samples_per_second': 20.835, 'train_steps_per_second': 5.209, 'total_flos': 6.375086531857613e+16, 'train_loss': 6.669971419078714, 'epoch': 3.0})

In [47]:
trainer.save_model("./final-nllb-mizo")
tokenizer.save_pretrained("./final-nllb-mizo")

print("Model saved successfully")

Model saved successfully


In [52]:
import torch

text = "I will come back tomorrow morning."

# Set source language
tokenizer.src_lang = "eng_Latn"

# Tokenize input
inputs = tokenizer(text, return_tensors="pt")

# Move tensors to GPU if available
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate translation
generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("lus_Latn"),
    max_length=128
)

# Decode
translation = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

print("Translation:")
print(translation[0])

Translation:
Ka lo kal leh dawn a ni .


In [53]:
!zip -r final-nllb-mizo.zip final-nllb-mizo

  adding: final-nllb-mizo/ (stored 0%)
  adding: final-nllb-mizo/tokenizer_config.json (deflated 75%)
  adding: final-nllb-mizo/adapter_model.safetensors (deflated 7%)
  adding: final-nllb-mizo/adapter_config.json (deflated 58%)
  adding: final-nllb-mizo/training_args.bin (deflated 53%)
  adding: final-nllb-mizo/README.md (deflated 66%)
  adding: final-nllb-mizo/tokenizer.json (deflated 82%)


In [54]:
from google.colab import files

files.download("final-nllb-mizo.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>